[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardoberta/regressione-superiori/blob/main/04_generalizzazione_overfitting.ipynb)

# Notebook 4 — Quando il modello si illude di essere bravo

Un modello può sembrare bravissimo sui dati che ha già visto. Ma il vero obiettivo del machine learning è prevedere bene su casi nuovi. In questo episodio introduciamo training set, test set, generalizzazione e overfitting.

In [ ]:
from IPython.display import HTML, display

def concept(title, body, kind="idea"):
    colors = {
        "idea": ("#e8f3ff", "#0b5394"),
        "math": ("#fff3cd", "#7a4d00"),
        "warning": ("#fdecea", "#a61b1b"),
        "task": ("#eaf7ea", "#226622"),
        "story": ("#f3e8ff", "#5b2a86")
    }
    bg, border = colors.get(kind, colors["idea"])
    display(HTML(f'''
    <div style="background:{bg}; border-left:6px solid {border}; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.45">
    <b style="color:{border}; font-size:18px">{title}</b><br>{body}
    </div>
    '''))

def reveal(title, body):
    display(HTML(f'''
    <details style="background:#f7f7f7; border:1px solid #ddd; padding:12px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.45">
      <summary style="cursor:pointer; font-weight:bold">{title}</summary>
      <div style="margin-top:10px">{body}</div>
    </details>
    '''))

In [ ]:
concept("Missione", "Separare i dati in allenamento e test, confrontare modelli diversi e capire quando un modello sta imparando una regola oppure sta memorizzando i dati.", "story")

In [ ]:
# Setup: imports e download del dataset (solo la prima volta)
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://www.dropbox.com/scl/fi/0l5n46qjwcd5moj3w7d8p/fifa.zip?rlkey=rcqhagvq5ttlvna5t5r3vn1bm&st=uzplzs5o&dl=1"
ZIP_PATH = Path("fifa.zip")
DATA_DIR = Path("fifa_data")

if not ZIP_PATH.exists():
    print("Scarico il dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

if not list(DATA_DIR.rglob("*.csv")):
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

csv_files = list(DATA_DIR.rglob("players_22.csv")) or list(DATA_DIR.rglob("*players*.csv"))
raw = pd.read_csv(csv_files[0], low_memory=False)
print(f"Dataset caricato: {raw.shape[0]} giocatori, {raw.shape[1]} colonne")
raw.head()


In [ ]:
# Creiamo una versione didattica piccola e pulita
wanted_columns = [
    "short_name", "age", "overall", "potential", "wage_eur", "value_eur",
    "club_name", "league_name", "player_positions"
]

available = [c for c in wanted_columns if c in raw.columns]
df = raw[available].copy()

# Manteniamo solo i giocatori con valore noto e positivo.
df = df.dropna(subset=["value_eur", "age", "overall", "potential"])
df = df[df["value_eur"] > 0]

# Riduciamo l'effetto dei super-outlier per i primi grafici didattici.
df = df[df["value_eur"] <= df["value_eur"].quantile(0.99)]

# Per leggibilita' useremo spesso il valore in milioni di euro.
df["value_mln_eur"] = df["value_eur"] / 1_000_000
if "wage_eur" in df.columns:
    df["wage_k_eur"] = df["wage_eur"] / 1_000

print("Forma del dataset didattico:", df.shape)
df.head(10)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
concept("Generalizzazione", "Un modello generalizza quando funziona bene su dati nuovi, non solo sui dati usati per addestrarlo. Questo e' il cuore del machine learning: non vogliamo una memoria dei casi passati, vogliamo una regola utile per il futuro.", "idea")

<div style="background:#fff3cd; border-left:6px solid #7a4d00; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.5">
<b style="color:#7a4d00; font-size:18px">Teoria &mdash; Errore di training vs errore atteso</b>

<p>Quello che vorremmo davvero misurare &egrave; quanto sbaglier&agrave; il modello sui <b>casi futuri</b>, cio&egrave; l'<b>errore atteso</b>:</p>

$$R(f) \;=\; \mathbb{E}_{(x, y)}\!\left[(y - f(x))^2\right]$$

<p>Non possiamo calcolarlo esattamente, perch&eacute; non conosciamo tutti i casi del futuro. Possiamo per&ograve; calcolare l'<b>errore empirico</b> sui dati che abbiamo:</p>

$$\hat{R}_{\text{train}}(f) \;=\; \frac{1}{n_{\text{train}}} \sum_{i \in \text{train}} (y_i - f(x_i))^2$$

<p>Il punto delicato: $\hat{R}_{\text{train}}$ pu&ograve; ingannarci. Un modello molto flessibile pu&ograve; renderlo quasi zero "imparando a memoria" il training set, restando per&ograve; pessimo sui dati nuovi. Per stimare onestamente $R(f)$ misuriamo l'errore su un <b>test set</b> mai visto durante l'addestramento:</p>

$$\hat{R}_{\text{test}}(f) \;=\; \frac{1}{n_{\text{test}}} \sum_{i \in \text{test}} (y_i - f(x_i))^2$$

<p>&Egrave; questo numero che dobbiamo guardare per giudicare se il modello generalizza.</p>
</div>


## Dividiamo i dati: training e test

In [ ]:
features = [c for c in ["age", "overall", "potential", "wage_eur"] if c in df.columns]
X = df[features]
y = df["value_mln_eur"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Esempi di training:", len(X_train))
print("Esempi di test:", len(X_test))

In [ ]:
concept("Training set e test set", "Il training set serve per imparare i parametri del modello. Il test set resta nascosto fino alla valutazione finale. Se il modello va bene anche sul test set, abbiamo piu' fiducia che abbia imparato una regola generale.", "math")

## Modello lineare valutato correttamente

In [ ]:
lin = LinearRegression()
lin.fit(X_train, y_train)

pred_train_lin = lin.predict(X_train)
pred_test_lin = lin.predict(X_test)

def metrics(y_true, y_pred):
    return {
        "MAE_mln": mean_absolute_error(y_true, y_pred),
        "RMSE_mln": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred)
    }

results = []
results.append({"modello": "Lineare", "dati": "training", **metrics(y_train, pred_train_lin)})
results.append({"modello": "Lineare", "dati": "test", **metrics(y_test, pred_test_lin)})
pd.DataFrame(results).round(3)

## Un modello molto flessibile: albero decisionale

In [ ]:
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

pred_train_tree = tree.predict(X_train)
pred_test_tree = tree.predict(X_test)

results.append({"modello": "Albero non limitato", "dati": "training", **metrics(y_train, pred_train_tree)})
results.append({"modello": "Albero non limitato", "dati": "test", **metrics(y_test, pred_test_tree)})
pd.DataFrame(results).round(3)

In [ ]:
concept("Overfitting", "Overfitting significa che il modello si adatta troppo ai dettagli del training set. In quel caso l'errore sul training puo' diventare molto basso, ma l'errore sul test resta alto o peggiora. E' come uno studente che impara a memoria gli esercizi svolti ma non sa risolverne uno nuovo.", "warning")

## Limitiamo la complessità dell'albero

In [ ]:
tree_small = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_small.fit(X_train, y_train)

pred_train_small = tree_small.predict(X_train)
pred_test_small = tree_small.predict(X_test)

results.append({"modello": "Albero max_depth=4", "dati": "training", **metrics(y_train, pred_train_small)})
results.append({"modello": "Albero max_depth=4", "dati": "test", **metrics(y_test, pred_test_small)})

pd.DataFrame(results).round(3)

In [ ]:
concept("Complessita' del modello", "Un modello troppo semplice puo' non catturare la struttura dei dati: underfitting. Un modello troppo complesso puo' catturare anche rumore e casi particolari: overfitting. L'obiettivo e' trovare un equilibrio che funzioni bene sui dati nuovi.", "math")

## Vediamo l'overfitting come una curva


<div style="background:#fff3cd; border-left:6px solid #7a4d00; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.5">
<b style="color:#7a4d00; font-size:18px">Teoria &mdash; La curva a U dell'overfitting</b>

<p>Aumentando la complessit&agrave; del modello (per esempio la profondit&agrave; massima dell'albero) succede tipicamente che:</p>

<ul>
<li>L'<b>errore di training</b> cala sempre: pi&ugrave; il modello &egrave; flessibile, pi&ugrave; si adatta ai dati visti.</li>
<li>L'<b>errore di test</b> prima cala (il modello impara la regola vera), poi <b>risale</b> (il modello impara il rumore del training).</li>
</ul>

<p>Il punto migliore &egrave; il <b>minimo della curva di test</b>. &Egrave; il classico equilibrio tra:</p>

<ul>
<li><b>Bias</b> (errore sistematico): un modello troppo semplice non riesce a catturare il vero pattern &mdash; <i>underfitting</i>.</li>
<li><b>Varianza</b>: un modello troppo flessibile cambia molto al variare dei dati di training &mdash; <i>overfitting</i>.</li>
</ul>
</div>


In [ ]:
# Per varie profondita' dell'albero, misuriamo l'errore su training e su test
profondita = list(range(1, 21))
err_train = []
err_test = []

for d in profondita:
    m = DecisionTreeRegressor(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    err_train.append(mean_absolute_error(y_train, m.predict(X_train)))
    err_test.append(mean_absolute_error(y_test, m.predict(X_test)))

plt.figure(figsize=(8,5))
plt.plot(profondita, err_train, "o-", label="Errore su training", color="#2a9d8f")
plt.plot(profondita, err_test, "o-", label="Errore su test", color="#e76f51")
plt.xlabel("Profondita' massima dell'albero")
plt.ylabel("MAE [milioni di euro]")
plt.title("Underfitting vs overfitting al crescere della complessita'")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

best_d = profondita[int(np.argmin(err_test))]
print(f"Profondita' con errore di test minimo: {best_d}")


## Previsioni finali su casi di test

In [ ]:
test_view = X_test.copy()
test_view["valore_reale"] = y_test
test_view["lineare"] = pred_test_lin
test_view["albero_non_limitato"] = pred_test_tree
test_view["albero_limitato"] = pred_test_small

# Recuperiamo i nomi dei giocatori corrispondenti
names = df.loc[test_view.index, ["short_name", "club_name", "player_positions"]]
final_view = pd.concat([names, test_view], axis=1)
final_view.sample(12, random_state=5).round(2)

In [ ]:
reveal("Discussione finale", "Quale modello scegliereste per aiutare davvero una societa' sportiva? Non basta dire quello con errore minore sul training. Bisogna guardare il test set e ragionare sulla stabilita' del modello.")

In [ ]:
reveal("Cosa abbiamo imparato nella serie", "Abbiamo trasformato una domanda concreta, quanto vale un calciatore?, in un problema di regressione. Abbiamo esplorato dati, costruito un primo modello, aggiunto variabili, misurato errori e infine distinto tra prestazione apparente e generalizzazione.")